# 🛡️ Day 04a: API Best Practices — Production-Ready Patterns

---

## 🎯 What You'll Master Today
- Error handling that doesn't leak internals
- Pagination patterns (offset, cursor, keyset)
- Rate limiting, idempotency keys, retry logic
- API documentation with OpenAPI/Swagger

---

## 🎭 Why This Matters

Anyone can build an API that works. Interviews test whether you can build an API that **doesn't break in production**. This notebook covers the patterns that separate junior "it works on my machine" from senior "it handles 10K req/sec with graceful degradation."

---

## 🚨 Error Handling — The Right Way

```
╔═══════════════════════════════════════════════════════════════════╗
║  ERROR HANDLING PRINCIPLES                                       ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  1. NEVER expose internal errors to clients                      ║
║     ❌ "psycopg2.IntegrityError: duplicate key"                 ║
║     ✅ "Email already registered"                                ║
║                                                                   ║
║  2. Use CONSISTENT error format across ALL endpoints             ║
║     {status, error: {code, message, details}}                    ║
║                                                                   ║
║  3. Use MACHINE-READABLE error codes                             ║
║     "VALIDATION_ERROR", "NOT_FOUND", "RATE_LIMITED"           ║
║                                                                   ║
║  4. Include FIELD-LEVEL details for validation errors            ║
║     [{field: "email", message: "Invalid format"}]              ║
║                                                                   ║
║  5. Log the FULL error server-side, send SAFE error to client   ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

In [ ]:
# ============================================================
# Standard Error Response Format
# ============================================================
import json
from datetime import datetime

class APIError(Exception):
    """Base class for all API errors."""
    def __init__(self, status_code, code, message, details=None):
        self.status_code = status_code
        self.code = code
        self.message = message
        self.details = details or []
    
    def to_response(self):
        return {
            "status": "error",
            "error": {
                "code": self.code,
                "message": self.message,
                "details": self.details,
                "timestamp": datetime.utcnow().isoformat()
            }
        }

# Specific error types
class NotFoundError(APIError):
    def __init__(self, resource, resource_id):
        super().__init__(404, "NOT_FOUND", f"{resource} with id {resource_id} not found")

class ValidationError(APIError):
    def __init__(self, details):
        super().__init__(422, "VALIDATION_ERROR", "Invalid input data", details)

class RateLimitError(APIError):
    def __init__(self, retry_after=60):
        super().__init__(429, "RATE_LIMITED", f"Too many requests. Retry after {retry_after}s")

# Usage
try:
    raise NotFoundError("User", 42)
except APIError as e:
    print(json.dumps(e.to_response(), indent=2))

In [ ]:
# ============================================================
# Validation errors with field-level details
# ============================================================

try:
    raise ValidationError([
        {"field": "email", "message": "Invalid email format"},
        {"field": "age", "message": "Must be between 1 and 150"},
        {"field": "password", "message": "Must be at least 8 characters"}
    ])
except APIError as e:
    print(json.dumps(e.to_response(), indent=2))

## 🔑 Idempotency Keys — Preventing Duplicate Operations

```
╔═══════════════════════════════════════════════════════════════════╗
║  IDEMPOTENCY KEYS                                                ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Problem: User clicks "Pay" twice → charged twice!              ║
║                                                                   ║
║  Solution: Client sends unique key with each request             ║
║                                                                   ║
║  POST /payments                                                   ║
║  Idempotency-Key: abc-123-unique-key                             ║
║  {"amount": 50.00, "currency": "USD"}                          ║
║                                                                   ║
║  Server checks: Have I seen this key before?                     ║
║  • No → Process payment, store result with key                   ║
║  • Yes → Return stored result (don't process again)             ║
║                                                                   ║
║  Used by: Stripe, PayPal, Amazon — any payment/critical API     ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

In [ ]:
# ============================================================
# Idempotency Key Implementation
# ============================================================
import uuid

# In-memory store (use Redis in production)
idempotency_store = {}

def process_payment(idempotency_key, amount):
    """Process payment with idempotency protection."""
    # Check if we've already processed this key
    if idempotency_key in idempotency_store:
        print(f"⚡ Returning cached result for key: {idempotency_key}")
        return idempotency_store[idempotency_key]
    
    # Process the payment
    result = {
        "payment_id": str(uuid.uuid4()),
        "amount": amount,
        "status": "completed"
    }
    
    # Store result with TTL (24 hours in production)
    idempotency_store[idempotency_key] = result
    print(f"✅ Payment processed: ${amount}")
    return result

# First call — processes payment
key = "user-123-order-456"
result1 = process_payment(key, 50.00)

# Second call with SAME key — returns cached result (no double charge!)
result2 = process_payment(key, 50.00)

# Same payment_id returned
print(f"\nSame result? {result1['payment_id'] == result2['payment_id']}")

## ⏱️ Rate Limiting — Protecting Your API

```
╔═══════════════════════════════════════════════════════════════════╗
║  RATE LIMITING ALGORITHMS                                        ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  1. FIXED WINDOW: 100 requests per minute, reset at :00         ║
║     ⚠️ Burst at window boundary (200 reqs in 2 seconds)        ║
║                                                                   ║
║  2. SLIDING WINDOW LOG: Track timestamp of each request         ║
║     ✅ Precise  ❌ Memory-heavy                                  ║
║                                                                   ║
║  3. TOKEN BUCKET: Bucket fills at constant rate                  ║
║     ✅ Allows bursts  ✅ Smooth rate                             ║
║     Used by: AWS API Gateway, Stripe                             ║
║                                                                   ║
║  4. SLIDING WINDOW COUNTER: Weighted average of windows         ║
║     ✅ Good balance of precision and memory                      ║
║                                                                   ║
║  RESPONSE HEADERS:                                                ║
║  X-RateLimit-Limit: 100                                          ║
║  X-RateLimit-Remaining: 42                                       ║
║  X-RateLimit-Reset: 1640000000 (Unix timestamp)                  ║
║  Retry-After: 30 (seconds, on 429)                               ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

In [ ]:
# ============================================================
# Token Bucket Rate Limiter — Interview-Ready Code
# ============================================================
import time

class TokenBucketRateLimiter:
    def __init__(self, rate, capacity):
        """
        rate: tokens added per second
        capacity: max tokens (burst size)
        """
        self.rate = rate
        self.capacity = capacity
        self.tokens = capacity  # start full
        self.last_refill = time.time()
    
    def _refill(self):
        """Add tokens based on elapsed time."""
        now = time.time()
        elapsed = now - self.last_refill
        self.tokens = min(
            self.capacity,
            self.tokens + elapsed * self.rate
        )
        self.last_refill = now
    
    def allow_request(self):
        """Check if request is allowed."""
        self._refill()
        if self.tokens >= 1:
            self.tokens -= 1
            return True
        return False

# Demo: 2 requests per second, burst of 5
limiter = TokenBucketRateLimiter(rate=2, capacity=5)

# First 5 requests succeed (burst)
for i in range(7):
    allowed = limiter.allow_request()
    print(f"Request {i+1}: {'✅ Allowed' if allowed else '❌ Rate Limited'}")

## 📖 API Documentation — OpenAPI / Swagger

```
╔═══════════════════════════════════════════════════════════════════╗
║  API DOCUMENTATION BEST PRACTICES                                ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  OpenAPI (formerly Swagger) is the standard for REST API docs.   ║
║                                                                   ║
║  FastAPI generates OpenAPI docs AUTOMATICALLY from your code!    ║
║  • /docs → Swagger UI (interactive)                              ║
║  • /redoc → ReDoc (beautiful read-only)                          ║
║  • /openapi.json → raw spec                                     ║
║                                                                   ║
║  What to include:                                                 ║
║  • All endpoints with methods                                     ║
║  • Request/response schemas (Pydantic does this for you)        ║
║  • Authentication requirements                                    ║
║  • Example requests and responses                                 ║
║  • Error codes and formats                                        ║
║  • Rate limiting info                                             ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

In [ ]:
# ============================================================
# Retry Logic with Exponential Backoff
# ============================================================
# Know this pattern for interviews — it's asked a LOT

import random
import time

def retry_with_backoff(func, max_retries=3, base_delay=1.0):
    """Retry with exponential backoff + jitter."""
    for attempt in range(max_retries + 1):
        try:
            return func()
        except Exception as e:
            if attempt == max_retries:
                raise  # Give up after max retries
            
            # Exponential backoff: 1s, 2s, 4s, 8s...
            delay = base_delay * (2 ** attempt)
            # Add jitter to prevent thundering herd
            jitter = random.uniform(0, delay * 0.1)
            total_delay = delay + jitter
            
            print(f"  Attempt {attempt + 1} failed: {e}")
            print(f"  Retrying in {total_delay:.2f}s...")
            # time.sleep(total_delay)  # would actually wait

# Simulate a flaky API
call_count = 0
def flaky_api():
    global call_count
    call_count += 1
    if call_count < 3:
        raise ConnectionError("Server unavailable")
    return {"status": "success"}

result = retry_with_backoff(flaky_api)
print(f"\n✅ Success on attempt {call_count}: {result}")

In [ ]:
# ============================================================
# Health Check Endpoint Pattern
# ============================================================
# Every production API needs these

import time

def health_check():
    """Basic health check — is the service alive?"""
    return {
        "status": "healthy",
        "timestamp": time.time()
    }

def readiness_check():
    """Readiness check — can it serve traffic?
    Check all dependencies."""
    checks = {
        "database": check_db(),
        "redis": check_redis(),
        "external_api": check_external()
    }
    all_healthy = all(c["status"] == "up" for c in checks.values())
    return {
        "status": "ready" if all_healthy else "not_ready",
        "checks": checks
    }

# Simulated dependency checks
def check_db(): return {"status": "up", "latency_ms": 5}
def check_redis(): return {"status": "up", "latency_ms": 1}
def check_external(): return {"status": "up", "latency_ms": 50}

import json
print("GET /health")
print(json.dumps(health_check(), indent=2))
print("\nGET /ready")
print(json.dumps(readiness_check(), indent=2))

---

## 🗺️ API Best Practices Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  API BEST PRACTICES QUICK REFERENCE                              ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  ERRORS: consistent format, machine-readable codes,              ║
║          field-level details, never expose internals              ║
║                                                                   ║
║  IDEMPOTENCY: Idempotency-Key header for POST /payments         ║
║               Store result → return cached on retry              ║
║                                                                   ║
║  RATE LIMITING: Token bucket (allows bursts)                     ║
║                 Return X-RateLimit-* headers + Retry-After       ║
║                                                                   ║
║  RETRY: Exponential backoff + jitter                             ║
║         Prevents thundering herd problem                         ║
║                                                                   ║
║  HEALTH: /health (liveness) + /ready (readiness)                ║
║  DOCS: OpenAPI auto-generated by FastAPI                         ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions — Practice These

| # | Question | Key Points |
|---|----------|-----------|
| 1 | How do you handle errors in a REST API? | Consistent error format with code + message + details. Never expose stack traces |
| 2 | What is an idempotency key? | Client-generated unique key to prevent duplicate operations (payments) |
| 3 | How do you implement rate limiting? | Token bucket for burst + steady rate. Return 429 + Retry-After |
| 4 | What is exponential backoff? | Double wait time on each retry (1s, 2s, 4s). Add jitter to prevent thundering herd |
| 5 | Health check vs readiness check? | Health: is process alive? Ready: can it handle traffic? (DB connected, etc.) |
| 6 | How do you document an API? | OpenAPI/Swagger. FastAPI generates it automatically from type hints |
| 7 | How to handle partial failures? | Circuit breaker pattern, graceful degradation, fallback responses |
| 8 | What security headers should an API return? | X-Content-Type-Options, X-Frame-Options, Strict-Transport-Security |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Consistent error format across all endpoints          │
## │  • Idempotency keys prevent double-charging              │
## │  • Token bucket rate limiter allows bursts               │
## │  • Exponential backoff + jitter for retries              │
## │  • /health + /ready endpoints are mandatory              │
## │  • Never expose internal errors to clients               │
## │  • FastAPI gives you free OpenAPI docs                   │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Hello FastAPI!** — Building your first FastAPI application from scratch

### 🎉 Chapter 01 Complete!
You now understand HTTP, REST design, WSGI/ASGI, and production-ready API patterns. Time to build with FastAPI!